# Análisis de Entidades Nombradas (NER) con Flair

Este notebook utiliza la librería [Flair](https://github.com/flairNLP/flair) para realizar el reconocimiento de entidades nombradas en las oraciones del archivo `ground_truth_ner.json`.

Se utilizará un modelo pre-entrenado para español.

In [9]:
# Instalación de Flair (si no está instalado)
!pip install flair

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 3.1 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 12.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 15.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 40.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.1/203.1 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.5/793.5 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━

In [1]:
import json
import pandas as pd
from flair.data import Sentence
from flair.models import SequenceTagger

# Cargar el modelo NER para español
# 'flair/ner-spanish-large' es un modelo basado en transformers (Beto) que suele dar muy buenos resultados.
print("Cargando modelo NER español...")
tagger = SequenceTagger.load('flair/ner-spanish-large')
print("Modelo cargado exitosamente.")

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando modelo NER español...
2026-02-16 04:33:57,627 SequenceTagger predicts: Dictionary with 20 tags: <unk>, O, S-LOC, S-ORG, B-PER, I-PER, E-PER, S-MISC, B-ORG, E-ORG, S-PER, I-ORG, B-LOC, E-LOC, B-MISC, E-MISC, I-MISC, I-LOC, <START>, <STOP>
Modelo cargado exitosamente.


In [2]:
# Cargar los datos del archivo JSON
file_path = 'ground_truth_ner.json'

with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"Se cargaron {len(data)} oraciones para analizar.")

Se cargaron 15 oraciones para analizar.


## Procesamiento de Oraciones

Iteramos sobre cada texto, creamos un objeto `Sentence` de Flair, predecimos las etiquetas y extraemos las entidades encontradas con su confianza.

In [3]:
results = []

for item in data:
    text = item['text']
    
    # Crear objeto Sentence
    sentence = Sentence(text)
    
    # Predecir etiquetas NER
    tagger.predict(sentence)
    
    # Extraer entidades
    # get_spans('ner') devuelve las entidades encontradas
    entities = []
    for entity in sentence.get_spans('ner'):
        entities.append({
            'text': entity.text,
            'label': entity.tag,
            'score': entity.score
        })
    
    # Guardar resultados
    results.append({
        'id': item['id'],
        'text': text,
        'entities': entities,
        'flair_sentence': sentence # Guardamos el objeto para inspección si es necesario
    })

# Crear DataFrame para visualizar
# Aplanamos la lista de entidades para mostrar mejor en tabla, o mostramos la lista completa
df_results = pd.DataFrame(results).drop(columns=['flair_sentence'])
pd.set_option('display.max_colwidth', None)
df_results

,id,text,entities
0,1,Genera una cotización para el cliente Compufacil con 5 monitores led y 3 soportes de pared,"[{'text': 'Compufacil', 'label': 'ORG', 'score': 0.999840259552002}]"
1,2,Prepara un presupuesto urgente con 10 teclados inalámbricos y 10 ratones ópticos para enviar a Tecnosys,"[{'text': 'Tecnosys', 'label': 'ORG', 'score': 0.999954104423523}]"
2,3,Crea una oferta comercial para Carla Santana con 1 escritorio ejecutivo modelo XG Premium,"[{'text': 'Carla Santana', 'label': 'PER', 'score': 0.9999377727508545}, {'text': 'XG Premium', 'label': 'MISC', 'score': 0.8730040490627289}]"
3,4,Genera una proforma para AndinaCorp incluye 2 laptops core i7 y 3 impresoras multifunción,"[{'text': 'AndinaCorp', 'label': 'ORG', 'score': 0.9999052286148071}]"
4,5,Busca la última factura del cliente Velasco y Asociados y reenvíala a su correo,"[{'text': 'Velasco y Asociados', 'label': 'ORG', 'score': 0.9565423329671224}]"
5,6,Genera la factura del pedido de compra 852025,[]
6,7,Verifica si la factura FA 409516 de Hierros del Pacífico ya está pagada,"[{'text': 'FA', 'label': 'MISC', 'score': 0.9946964979171753}, {'text': 'Hierros del Pacífico', 'label': 'LOC', 'score': 0.9998737374941508}]"
7,8,¿Cuántas sillas ergonómicas de oficina tenemos en la bodega de Guayaquil?,"[{'text': 'Guayaquil', 'label': 'LOC', 'score': 0.9999837875366211}]"
8,9,Registra el ingreso de 50 resmas de papel A4 del proveedor Papel Mundo,"[{'text': 'Papel Mundo', 'label': 'ORG', 'score': 0.9999228119850159}]"
9,10,Dame el stock actual de discos duros de 1 terabyte de la marca Duradisco,"[{'text': 'Duradisco', 'label': 'ORG', 'score': 0.9973570108413696}]"


## Visualización Detallada

Mostramos las oraciones con sus entidades resaltadas de una forma legible.

In [4]:
for item in results:
    print(f"ID: {item['id']}")
    print(f"Texto: {item['text']}")
    if item['entities']:
        print("Entidades encontradas:")
        for ent in item['entities']:
            print(f"  - {ent['text']} ({ent['label']}) [Confianza: {ent['score']:.4f}]")
    else:
        print("  (No se encontraron entidades)")
    print("-" * 50)

ID: 1
Texto: Genera una cotización para el cliente Compufacil con 5 monitores led y 3 soportes de pared
Entidades encontradas:
  - Compufacil (ORG) [Confianza: 0.9998]
--------------------------------------------------
ID: 2
Texto: Prepara un presupuesto urgente con 10 teclados inalámbricos y 10 ratones ópticos para enviar a Tecnosys
Entidades encontradas:
  - Tecnosys (ORG) [Confianza: 1.0000]
--------------------------------------------------
ID: 3
Texto: Crea una oferta comercial para Carla Santana con 1 escritorio ejecutivo modelo XG Premium
Entidades encontradas:
  - Carla Santana (PER) [Confianza: 0.9999]
  - XG Premium (MISC) [Confianza: 0.8730]
--------------------------------------------------
ID: 4
Texto: Genera una proforma para AndinaCorp incluye 2 laptops core i7 y 3 impresoras multifunción
Entidades encontradas:
  - AndinaCorp (ORG) [Confianza: 0.9999]
--------------------------------------------------
ID: 5
Texto: Busca la última factura del cliente Velasco y Asociados y